# Dockerizing Our Data Ingestion Pipeline

So far, you have written a Python script (`ingest.py`) that extracts live Bitcoin transactions and loads them into a PostgreSQL database. However, this script currently runs directly on your local machine. If a teammate tries to run it, they might encounter issues with Python versions, missing packages, or OS mismatches.

In this module, you will learn the concept of **Dockerizing**,packaging our application, dependencies, and environment into a single container that runs identically anywhere.

## 1. What is Dockerizing?

Imagine you are shipping a complex piece of machinery to a customer. Instead of sending loose parts, tools, and an instruction manual, you ship a fully assembled machine inside a shipping container. Once it arrives, the customer just plugs it in, and it works.

**Dockerizing** is the process of creating a container for your code. It bundles:
1. **The Operating System:** A minimal, lightweight Linux distribution.
2. **The Runtime:** Python and tools like `uv`.
3. **The Dependencies:** Exact versions of libraries (Pandas, SQLAlchemy, Psycopg) locked in your environment.
4. **The Code:** Your `ingest.py` script.

## 2. The Blueprint: The Dockerfile

To build a container, we write a recipe file named `Dockerfile` (no file extension). Here is the configuration we will use:

```bash
FROM python:3.11-slim
COPY --from=ghcr.io/astral-sh/uv:latest /uv /bin/

WORKDIR /code
ENV PATH="/code/.venv/bin:$PATH"

COPY pyproject.toml .python-version uv.lock ./
RUN uv sync --locked

COPY ingest.py .

ENTRYPOINT ["uv", "run", "python", "ingest.py"]
```

### What does this Dockerfile do?

- **`FROM python:3.11-slim`**:
  Starts with a lightweight, secure Linux image that has Python 3.11 pre-installed.
- **`COPY --from=ghcr.io/astral-sh/uv:latest /uv /bin/`**:
  Downloads and installs **`uv`**, the ultra-fast Python package installer and manager.
- **`WORKDIR /code`**:
  Creates and sets `/code` as the default working directory inside the container.
- **`ENV PATH="/code/.venv/bin:$PATH"`**:
  Adds the container's virtual environment path to the system path, ensuring Python executes packages from our environment.
- **`COPY pyproject.toml .python-version uv.lock ./`**:
  Copies our environment configurations and package lockfiles first. By copying configuration before our source code, Docker can cache our installed dependencies. If we change our code later, we don't have to wait for packages to download again!
- **`RUN uv sync --locked`**:
  Installs the exact locked dependencies defined in our `uv.lock` file.
- **`COPY ingest.py .`**:
  Copies our data ingestion script into the container.
- **`ENTRYPOINT ["uv", "run", "python", "ingest.py"]`**:
  Defines the default command that runs when the container starts. In this case, it executes our ingestion script.

## 3. Building and Running the Container

Open your terminal in the directory containing the `Dockerfile` and run the following commands:

### Step 1: Build the Image
We compile our Dockerfile recipe into an executable image named `bitcoin-ingest`:
```bash
docker build -t bitcoin-ingest .
```

### Step 2: Run the Ingestion Pipeline
Because the database is running on your local machine (`localhost`), we run the container using the host's network namespace so that the container can access `localhost:5432`:
```bash
docker run --network="host" bitcoin-ingest
```

Once it runs, you will see the logs showing live transaction fetching and successful loading into PostgreSQL, all running inside an isolated container!